# Course Project Colab Server

Upload this notebook to Colab and click **Run all**.

It will:
1. install the serving dependencies
2. start a GPU-backed OpenAI-compatible model server
3. expose it through an ephemeral public tunnel
4. print a copy-paste `export ...` block for your local terminal

After the final cell, go back to your laptop and run:

```bash
cd course_project
bash launch
```

In [ ]:
import os
import subprocess
import sys


def run(cmd):
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)


run([sys.executable, "-m", "pip", "install", "-q", "vllm", "requests"])

CLOUDFLARED_BIN = "/content/cloudflared"
if not os.path.exists(CLOUDFLARED_BIN):
    run([
        "bash",
        "-lc",
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared",
    ])

print("Install step complete.")
print("cloudflared:", CLOUDFLARED_BIN)

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=False)

MODEL_PRESETS = {
    "baseline_7b": [
        {
            "model_id": "Qwen/Qwen2.5-7B-Instruct-AWQ",
            "served_model_name": "qwen2.5-7b-awq",
        },
        {
            "model_id": "Qwen/Qwen2.5-3B-Instruct",
            "served_model_name": "qwen2.5-3b-instruct",
        },
    ],
    "stronger_14b": [
        {
            "model_id": "Qwen/Qwen2.5-14B-Instruct-AWQ",
            "served_model_name": "qwen2.5-14b-awq",
        },
        {
            "model_id": "Qwen/Qwen2.5-7B-Instruct-AWQ",
            "served_model_name": "qwen2.5-7b-awq",
        },
    ],
}

ACTIVE_MODEL_PRESET = "baseline_7b"
MODEL_CANDIDATES = MODEL_PRESETS[ACTIVE_MODEL_PRESET]

PORT = 8000
API_KEY = ""
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.90
SERVER_LOG = "/content/vllm_server.log"
TUNNEL_LOG = "/content/cloudflared.log"

print("Active model preset:", ACTIVE_MODEL_PRESET)
print("Model candidates:")
for item in MODEL_CANDIDATES:
    print("-", item["model_id"])
print("Port:", PORT)
print("API key auth:", "disabled" if not API_KEY else "enabled")

In [ ]:
import os
import subprocess
import time

import requests


def tail_text(path, n=80):
    if not os.path.exists(path):
        return ""
    with open(path, "r", encoding="utf-8", errors="replace") as handle:
        lines = handle.readlines()
    return "".join(lines[-n:])


def stop_process(process):
    if process is None:
        return
    if process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=15)


def wait_for_server(base_url, api_key, process, timeout_seconds=900):
    headers = {}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        if process.poll() is not None:
            raise RuntimeError(
                "vLLM server exited before becoming ready.\n\n" + tail_text(SERVER_LOG)
            )
        try:
            response = requests.get(f"{base_url}/v1/models", headers=headers, timeout=15)
            if response.ok:
                return
            last_error = f"HTTP {response.status_code}: {response.text}"
        except Exception as exc:
            last_error = str(exc)
        time.sleep(5)
    raise RuntimeError(f"Timed out waiting for vLLM server. Last error: {last_error}")


server_process = None
selected_model = None

for candidate in MODEL_CANDIDATES:
    if os.path.exists(SERVER_LOG):
        os.remove(SERVER_LOG)
    log_handle = open(SERVER_LOG, "w", encoding="utf-8")
    command = [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        candidate["model_id"],
        "--served-model-name",
        candidate["served_model_name"],
        "--host",
        "0.0.0.0",
        "--port",
        str(PORT),
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
    ]
    if API_KEY:
        command.extend(["--api-key", API_KEY])
    print("Starting:", candidate["model_id"])
    server_process = subprocess.Popen(command, stdout=log_handle, stderr=subprocess.STDOUT)
    try:
        wait_for_server(f"http://127.0.0.1:{PORT}", API_KEY, server_process)
        selected_model = candidate
        log_handle.close()
        break
    except Exception as exc:
        print("Server start failed for", candidate["model_id"])
        print(str(exc))
        stop_process(server_process)
        log_handle.close()
        server_process = None

if selected_model is None:
    raise RuntimeError("No model candidate started successfully.")

print("Server ready.")
print("Selected model:", selected_model)

In [ ]:
import os
import re
import subprocess
import time


def stop_process(process):
    if process is None:
        return
    if process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=15)


if os.path.exists(TUNNEL_LOG):
    os.remove(TUNNEL_LOG)

tunnel_log = open(TUNNEL_LOG, "w", encoding="utf-8")
tunnel_process = subprocess.Popen(
    [CLOUDFLARED_BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
pattern = re.compile(r"https://[-a-zA-Z0-9.]+trycloudflare.com")
deadline = time.time() + 180

while time.time() < deadline and public_url is None:
    if tunnel_process.poll() is not None:
        tunnel_log.close()
        raise RuntimeError("cloudflared exited before producing a public URL.")
    line = tunnel_process.stdout.readline()
    if not line:
        time.sleep(1)
        continue
    tunnel_log.write(line)
    tunnel_log.flush()
    print(line, end="")
    match = pattern.search(line)
    if match:
        public_url = match.group(0)

tunnel_log.close()

if public_url is None:
    stop_process(tunnel_process)
    raise RuntimeError("Could not find a public tunnel URL.")

BASE_URL = public_url.rstrip("/") + "/v1"
print("Public base URL:", BASE_URL)

In [ ]:
import socket
import time
from urllib.parse import urlparse

import requests


def wait_for_public_endpoint(base_url, headers, timeout_seconds=180):
    parsed = urlparse(base_url)
    host = parsed.hostname
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        if tunnel_process.poll() is not None:
            raise RuntimeError("cloudflared exited before the public endpoint became ready.")
        try:
            socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
        except OSError as exc:
            last_error = f"DNS not ready: {exc}"
            time.sleep(3)
            continue
        try:
            response = requests.get(f"{base_url}/models", headers=headers, timeout=30)
            if response.ok:
                return
            last_error = f"HTTP {response.status_code}: {response.text[:200]}"
        except Exception as exc:
            last_error = str(exc)
        time.sleep(3)
    raise RuntimeError(f"Public tunnel did not become ready. Last error: {last_error}")

headers = {
    "Content-Type": "application/json",
}
if API_KEY:
    headers["Authorization"] = f"Bearer {API_KEY}"
payload = {
    "model": selected_model["served_model_name"],
    "messages": [{"role": "user", "content": "Reply with READY."}],
    "max_tokens": 32,
    "temperature": 0,
}

wait_for_public_endpoint(BASE_URL, headers)

response = requests.post(
    f"{BASE_URL}/chat/completions",
    headers=headers,
    json=payload,
    timeout=120,
)
response.raise_for_status()
data = response.json()
print("Smoke test response:")
print(data["choices"][0]["message"]["content"])


In [ ]:
import textwrap

export_block = textwrap.dedent(
    f"""
    export LLAMACPP_BASE_URL='{BASE_URL}'
    export LLAMACPP_MODEL='{selected_model['served_model_name']}'
    export LLAMACPP_API_KEY='{API_KEY}'
    export LOCAL_MODEL_PRESET='{ACTIVE_MODEL_PRESET}'
    export LOCAL_CONTEXT_WINDOW='{MAX_MODEL_LEN}'
    export LOCAL_TIMEOUT_SECONDS='300'
    export LOCAL_MAX_TOKENS='512'
    """
).strip()

with open('/content/personal_agent_env.sh', 'w', encoding='utf-8') as handle:
    handle.write(export_block + "\n")

with open('/content/.env.colab', 'w', encoding='utf-8') as handle:
    handle.write(export_block + "\n")

print("Copy this exact block into course_project/.env.colab in your local repo:\n")
print(export_block)
print("\nThen run locally:\n")
print("cd course_project")
print("bash launch")
print("\nThen run the visible Stage-1 benchmark:\n")
print("bash launch eval")